# Sprint 1 · Inspect and extend the planner application

This optional companion uses the application's actual functions. Complete the [local setup](https://github.com/richhiey/ml-llm-ops/blob/main/docs/setup.md) and work in the same clone as the Campus source/terminal route. The existing feature guard remains correct. Your real change adds a complete planned-file validation stage, its delivery integration and tests.

Save source edits in VS Code. Independent cells stay unsolved: predict first, probe actual inputs, then open feedback. Keep your baseline and earlier failures when a later source revision creates new evidence.

In [ ]:
from pathlib import Path
import json, sys, subprocess, runpy, importlib
import pandas as pd
import numpy as np
import mlflow.sklearn
from linden_mlops import Config, Workspace, ingest, preprocess
import linden_mlops.lifecycle as lifecycle
from linden_mlops.lifecycle import ROOT, code_revision
from linden_mlops.pipeline import DATA
from linden_mlops.visuals import plot_contract, save_figure
from IPython.display import display
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Use the project's locked Python 3.12 environment.")
work = ROOT / 'work'
work.mkdir(exist_ok=True)
workspace = Workspace(work / 'baseline-state')
print('Imported application source:', ROOT)
print('Notebook interpreter:', sys.executable)
print('Loaded source revision:', code_revision())

## A-S1-01 · From notebook to pipeline

Draw the historical fitting/evaluation and planned inference paths. Mark where known demand is required and which learned objects travel with the predictor. A script with hidden state has the same dependency problem as a notebook.

### Learning outcome

Explain why a reproducible training/inference pipeline replaces a one-off notebook workflow, and identify the 5 stages a production ML pipeline must cover - ingestion, preprocessing, training, evaluation, inference.

[Campus explanation, task and feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/01-from-notebook-to-pipeline/lesson.md).

## A-S1-02 · Reading the provided pipeline scaffold

Read `pipeline.py` and `lifecycle.py`. Follow core ingestion, preprocessing, fitting/evaluation and batch delivery. The cell below records a real baseline once, then reopens that manifest. It passes a fitted artifact URI directly; no registry policy is hidden inside a demonstration command.

### Learning outcome

Trace how data moves through the provided training/inference pipeline scaffold from raw input to served prediction, and label each stage's purpose.

[Campus explanation, task and feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/02-reading-the-provided-pipeline-scaffold/lesson.md).

In [ ]:
config = Config(alpha=1, range_guard=True, autolog=True)
record_path = work / 'baseline-run.json'
if record_path.exists():
    baseline = json.loads(record_path.read_text())
else:
    baseline = workspace.run(config, name='planner-baseline')
    record_path.write_text(json.dumps(baseline, indent=2) + '\n')
frame = ingest()
features = preprocess(frame, config)
fitted = mlflow.sklearn.load_model(baseline['model_uri'])
planned = pd.read_csv(DATA / 'batch-v1.csv')
print('Partitions, counts only:', frame.groupby('partition').size().to_dict())
print('Feature order:', features.columns.tolist())
print('Fitted steps:', list(fitted.named_steps))
print('Scaler fitting rows:', fitted.named_steps['scale'].n_samples_seen_)
print('Validation:', baseline['metrics'])
display(frame.loc[frame.partition.eq('train')].head(1))
display(planned.head(1))
path = workspace.path / 'baseline-batch.json'
if path.exists():
    baseline_delivery = json.loads(path.read_text())
    baseline_output = pd.read_csv(workspace.path / 'baseline-batch.csv')
else:
    baseline_output, baseline_delivery = lifecycle.batch_predict(
        workspace, output_name='baseline-batch', model_uri=baseline['model_uri'])
assert baseline_output.row_id.tolist() == planned.row_id.tolist()
assert len(baseline_output) == 24 and np.isfinite(baseline_output.prediction_kwh).all()
assert baseline_delivery['run_id'] == baseline['run_id']
assert baseline_delivery['complete_within_60s']
print('Actual delivery:', baseline_delivery)

## A-S1-03 · Adding and validating a bounded pipeline stage

The planner requires exactly 24 rows with the declared schema and complete unique nonblank identifiers. The current application can predict a shorter file. Follow the expert in the Campus walkthrough: author `tests/test_batch_input.py`, observe its intended acceptance failure, then add the new `src/linden_mlops/batch_input.py` stage and its explicit call in `batch_predict()`. Do not remove or weaken the existing range guard.

### Learning outcome

Follow an expert adding one preprocessing or validation stage to the provided pipeline, then verify its input/output contract and confirm that the unchanged stages still run.

[Campus explanation, task and feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/03-adding-and-validating-a-bounded-pipeline-stage/lesson.md).

### Observe the unmet requirement

Write `test_partial_plan_is_rejected` from the walkthrough into `tests/test_batch_input.py`, then run this cell before adding the stage. Expect `DID NOT RAISE`: a 23-row file previously reached delivery. A setup/import error does not establish that behavior. If you already completed the real edit, preserve and inspect the original `intake-before.txt` instead of undoing it.

In [ ]:
if not (ROOT / 'tests/test_batch_input.py').exists():
    raise RuntimeError('Write the acceptance test before continuing.')
if not (ROOT / 'src/linden_mlops/batch_input.py').exists():
    observed = subprocess.run([
        sys.executable, '-m', 'pytest',
        'tests/test_batch_input.py::test_partial_plan_is_rejected', '-q'],
        cwd=ROOT, capture_output=True, text=True)
    print(observed.stdout)
    (work / 'intake-before.txt').write_text(observed.stdout + observed.stderr)
    assert observed.returncode == 1 and 'DID NOT RAISE' in observed.stdout
else:
    print('Existing stage left intact. Keep your original failed-before observation.')

### Add the new stage and integration in VS Code

Create `batch_input.py`, add the import and call in `lifecycle.py`, and add preservation/boundary tests from the walkthrough. Explain why a valid row is not a complete daily file, why unit success alone cannot prove integration, and why the returned table is independent. Inspect the actual new files and lifecycle diff in Source Control. New untracked files do not appear in ordinary `git diff`.

In [ ]:
# Save your new module, lifecycle call and learner-owned tests before continuing.

In [ ]:
if not (ROOT / 'src/linden_mlops/batch_input.py').exists():
    raise RuntimeError('Add the real intake stage and integration before verification.')
for arguments, evidence_name in [
    (['tests/test_batch_input.py', '-q'], 'intake-after.txt'),
    (['tests/test_spine.py::test_contract_extension_and_unchanged_pipeline', '-q'], 'pipeline-regression.txt')]:
    checked = subprocess.run([sys.executable, '-m', 'pytest', *arguments],
                             cwd=ROOT, capture_output=True, text=True)
    print(checked.stdout)
    (work / evidence_name).write_text(checked.stdout + checked.stderr)
    if checked.returncode:
        raise RuntimeError('Inspect the failing saved-source test before claiming completion.')
# Reload saved lifecycle source for this notebook's direct application call.
lifecycle = importlib.reload(lifecycle)
workspace = lifecycle.Workspace(work / 'baseline-state')
delivery_name = 'validated-batch'  # choose a fresh name after a further source repair
metadata_path = workspace.path / f'{delivery_name}.json'
if metadata_path.exists():
    completed_delivery = json.loads(metadata_path.read_text())
    if completed_delivery.get('serving_code_revision') != lifecycle.code_revision():
        raise RuntimeError('Retain the old job; choose a new delivery_name for changed serving source.')
    completed_output = pd.read_csv(workspace.path / f'{delivery_name}.csv')
else:
    completed_output, completed_delivery = lifecycle.batch_predict(
        workspace, output_name=delivery_name, model_uri=baseline['model_uri'])
assert completed_output.row_id.tolist() == planned.row_id.tolist()
assert len(completed_output) == 24 and np.isfinite(completed_output.prediction_kwh).all()
assert completed_delivery['run_id'] == baseline['run_id']
assert completed_delivery['complete_within_60s']
print('Verified serving-source change:', completed_delivery)
print('Original fitted-model training revision:', baseline['code_revision'])

### Keep existing feature validation separate

The figure below probes the application's pre-existing domain guard. It is regression evidence for the unchanged feature boundary, not a picture of your newly added complete-file stage. Its partition panel reads final-row counts only; no final targets or scores are inspected.

In [ ]:
contract_figure = plot_contract(frame)
display(contract_figure)
print(contract_figure.linden_data)
save_figure(contract_figure, work / 'figures/existing-domain-contract')

## A-S1-04 · Explain-the-code checkpoint

Review the deliberately unsafe assistant-style domain-guard patch outside installed source. Explain each executable line, predict valid/out-of-range/negative/fractional-hour behavior, then probe fresh inputs. Keep returned values, caller preservation and object identity separate. This authored fixture is not a new assistant interaction.

### Learning outcome

Given an AI-generated addition to the pipeline, explain line by line what it does, test its input/output contract, and identify any part that needs correction before merging.

[Campus explanation, task and feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/04-explain-the-code-checkpoint/lesson.md).

In [ ]:
candidate_path = ROOT / 'docs/course/exercises/generated-stage-candidate.py'
print(candidate_path.read_text())
proposed_guard = runpy.run_path(str(candidate_path))['proposed_guard']
from linden_mlops.pipeline import validate_ranges as project_guard

In [ ]:
# Write your line explanations, predictions, fresh-input probes and merge recommendation here.

<details><summary>Solution — compare after your review and probes</summary>

**Result you should get:** The candidate clips upper-hour violations in the caller, misses negative/fractional hours and returns the original object. The correct existing guard rejects invalid inputs and preserves valid input in an independent table.

**How:** Retain a pre-call deep copy, test values and identity separately, and contrast the same invalid input with `project_guard`. The Campus feedback provides a concrete failed assertion. Preserve your own failed prediction and correct the explanation before requesting a merge.

</details>

## A-S1-05 · Batch vs real-time inference tradeoffs

Use `completed_delivery` and its actual CSV. Justify a pattern from shared arrival, the whole-file deadline, observed throughput and resource obligations. Explain what changes under a hypothetical individual 200 ms response budget. No endpoint latency or price is supplied.

### Learning outcome

Compare batch and real-time inference patterns - latency, cost, throughput - and justify which pattern the provided inference brief requires.

[Campus explanation, task and feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/05-batch-vs-real-time-inference-tradeoffs/lesson.md).

In [ ]:
# Record your recommendation and changed-brief measurement requirements here or in the same notes.

## A-S1-06 · Pipeline extension deliverable

Harbor supplies five unfamiliar feature-domain probes and three planned-file intakes. Predict each row/file verdict and responsible boundary before execution. Test your new stage and existing guard separately, then deliver an accepted complete file through the integrated application with a fresh client output name. Retain your source/test change, generated-code review and inference recommendation.

### Learning outcome

Deliverable: the provided pipeline scaffold extended with at least 1 validated stage, a line-by-line explanation of the changed logic, and a justified batch-vs-real-time recommendation for the inference brief.

[Campus explanation, task and feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-1/06-pipeline-extension-deliverable/lesson.md).

In [ ]:
supplier = pd.read_csv(ROOT / 'docs/course/exercises/harbor-intake.csv')
supplier_original = supplier.copy(deep=True)
client_plans = {label: pd.read_csv(ROOT / f'docs/course/exercises/harbor-plan-{label}.csv')
                for label in ['a', 'b', 'c']}
display(supplier)
for label, intake in client_plans.items():
    print('Client plan', label)
    display(intake)

In [ ]:
# Predict row/file decisions before execution, then author your actual probes.
# Check original preservation and independent returned identity.
# Invoke lifecycle.batch_predict with the baseline model_uri for an accepted complete plan,
# a new output name, and independently inspect its CSV/metadata.
# Keep the new-stage tests/diff, generated-code correction and inference justification.

<details><summary>Solution — compare after classifying and testing the client input</summary>

**Result you should get:** Plan A has 24 unique complete identifiers and passes unchanged; plan B repeats an identifier and rejects; plan C has 23 rows and rejects. A new client job from plan A preserves its input identifiers and supplies 24 finite predictions within 60 seconds. Separately H-201/H-203 pass the existing domain guard, while H-202/H-204/H-205 violate hour/temperature/occupancy. The five-row file is incomplete as a daily plan.

**How:** Call `validate_planned_batch` for file shape/schema/identity and `project_guard` for individual feature probes. Compare original tables after both acceptance and rejection. Invoke the actual delivery function for the complete accepted file; a direct unit-call result alone cannot prove integration. Use your own output name and actual metadata.

</details>

Keep the same completed source for the next controlled experiment. Its training runs will record that new application revision. The earlier fitted baseline remains an earlier training artifact, honestly identified.